# 🛰️ AntarikshaVaani - Dedicated ISRO Spacecraft LoRA Model Trainer
### By Team Stackverse-labs (Dayananda Sagar University)

This notebook trains a custom **ISRO Spacecraft LoRA** on your dataset for 100% mechanical & CAD accuracy.

**How to use:** Run the 4 cells in order by clicking the **Play `[▶]`** button on each.

### Step 1: Install AI Dependencies (~30s)

In [ ]:
# Step 1: Clean install of AI libraries
!pip install -q diffusers transformers accelerate peft torchvision safetensors huggingface_hub
print('✅ Step 1 SUCCESS: All AI libraries are ready!')

### Step 2: Load ISRO Dataset from GitHub (~3s)

In [ ]:
# Step 2: Pull dataset from your Stackverse repo
!rm -rf /content/dataset /content/repo
!git clone https://github.com/Omkar2005494/antarikshavaani.git /content/repo
!mkdir -p /content/dataset
!cp /content/repo/backend/data/lora_training/images/* /content/dataset/
!cp /content/repo/backend/data/lora_training/captions/* /content/dataset/

import os
files = [f for f in os.listdir('/content/dataset') if f.endswith(('.jpg', '.png'))]
print(f'✅ Step 2 SUCCESS: Loaded {len(files)} authentic spacecraft reference images!')
for f in files:
    print('  •', f)

### Step 3: Train the Model (~5 mins on Free T4 GPU)

In [ ]:
# Step 3: Self-contained training loop
import os
import torch
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import StableDiffusionPipeline
from peft import LoraConfig, get_peft_model

print('🚀 Loading base diffusion model...')
model_id = 'runwayml/stable-diffusion-v1-5'

class ISRODataset(Dataset):
    def __init__(self, folder):
        self.images = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.jpg', '.png'))]
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return self.transform(img)

dataset = ISRODataset('/content/dataset')
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Load pipeline & apply LoRA
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to('cuda')
unet = pipe.unet

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none'
)
unet = get_peft_model(unet, lora_config)
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

print('⚡ Starting training on GPU (300 steps)...')
unet.train()
steps = 0
total_steps = 300

while steps < total_steps:
    for batch in dataloader:
        if steps >= total_steps:
            break
        images = batch.to('cuda', dtype=torch.float16)
        latents = pipe.vae.encode(images).latent_dist.sample() * 0.18215
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, 1000, (1,), device='cuda').long()
        noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
        
        encoder_hidden_states = pipe.text_encoder(
            pipe.tokenizer('authentic isro spacecraft photograph', return_tensors='pt').input_ids.to('cuda')
        )[0]
        
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = torch.nn.functional.mse_loss(noise_pred, noise)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        steps += 1
        if steps % 50 == 0:
            print(f'  • Progress: [{steps}/{total_steps}] steps completed | Loss: {loss.item():.4f}')

# Save trained weights
os.makedirs('/content/isro_lora_output', exist_ok=True)
unet.save_pretrained('/content/isro_lora_output')
print('🎉 Step 3 SUCCESS: Training complete! Model saved!')

### Step 4: Download Your Trained Model File

In [ ]:
# Step 4: Auto-download the trained LoRA file to your computer
from google.colab import files
import os

target_file = None
for root, dirs, f_list in os.walk('/content/isro_lora_output'):
    for f in f_list:
        if f.endswith(('.safetensors', '.bin')):
            target_file = os.path.join(root, f)
            break

if target_file and os.path.exists(target_file):
    print(f'⬇️ Downloading {os.path.basename(target_file)} to your computer...')
    files.download(target_file)
else:
    print('⚠️ Please check /content/isro_lora_output for saved weights.')